# Stage 5: Reporting & Insight Generation

**Purpose**: Understand how Stage 5 transforms `Stage4Output` (refined chart data) into `PipelineResult` (final output with insights, reports, summaries).

**What this notebook covers**:
1. Construct realistic `Stage4Output` with mock data
2. Run `Stage5Reporting.process()` and inspect the full `PipelineResult`
3. Explore insight generation: trend detection, comparison, anomaly, summary
4. Inspect output formats: JSON, Markdown, CSV
5. Tune `ReportingConfig` parameters and observe effects

**Data Contract**:
```
Stage4Output (RefinedChartData[]) --> Stage5Reporting --> PipelineResult (FinalChartResult[] + insights)
```

## 0. Setup

In [ ]:
import sys
import json
from pathlib import Path
from datetime import datetime
from pprint import pprint

# Add project root to path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# Import schemas and stage
from src.core_engine.schemas.common import SessionInfo, BoundingBox, Point, Color
from src.core_engine.schemas.enums import ChartType, InsightType
from src.core_engine.schemas.stage_outputs import (
    Stage4Output,
    RefinedChartData,
    DataSeries,
    DataPoint,
    PipelineResult,
    FinalChartResult,
    ChartInsight,
)
from src.core_engine.stages.s5_reporting import Stage5Reporting, ReportingConfig

print("All imports successful")

## 1. Construct Realistic Stage4Output

Stage 4 produces `Stage4Output` containing a list of `RefinedChartData`.
Each `RefinedChartData` has:
- `chart_id`, `chart_type`, `title`, axis labels
- `series`: List of `DataSeries`, each with `DataPoint` objects
- `description`: Academic-style text description
- `correction_log`: OCR corrections applied by the reasoning engine

In [ ]:
# -- Helper: Create SessionInfo --
import hashlib

session = SessionInfo(
    session_id="nb05_demo_001",
    created_at=datetime.now(),
    source_file=Path("data/samples/test_bar_chart.png"),
    total_pages=1,
    config_hash=hashlib.md5(b"demo_config").hexdigest()[:16],
)
print(f"Session: {session.session_id}")
print(f"Source:  {session.source_file}")
print(f"Config:  {session.config_hash}")

In [ ]:
# -- Chart 1: Bar chart with clear trend (increasing revenue) --
bar_chart = RefinedChartData(
    chart_id="chart_001",
    chart_type=ChartType.BAR,
    title="Quarterly Revenue 2025",
    x_axis_label="Quarter",
    y_axis_label="Revenue (Million USD)",
    series=[
        DataSeries(
            name="Revenue",
            color=Color(r=66, g=133, b=244),
            points=[
                DataPoint(label="Q1", value=12.5, unit="M USD", confidence=0.95),
                DataPoint(label="Q2", value=15.3, unit="M USD", confidence=0.92),
                DataPoint(label="Q3", value=18.7, unit="M USD", confidence=0.88),
                DataPoint(label="Q4", value=22.1, unit="M USD", confidence=0.91),
            ],
        ),
    ],
    description="Bar chart showing quarterly revenue for 2025 with a consistent upward trend from Q1 (12.5M) to Q4 (22.1M USD).",
    correction_log=["Corrected 'Q1' from OCR 'Ql'", "Fixed decimal: '15.3' from '153'"],
)

print(f"Chart 1: {bar_chart.title}")
print(f"  Type: {bar_chart.chart_type.value}")
print(f"  Series: {len(bar_chart.series)}, Points: {bar_chart.series[0].count}")
print(f"  Corrections: {len(bar_chart.correction_log)}")

In [ ]:
# -- Chart 2: Line chart with multiple series + anomaly --
line_chart = RefinedChartData(
    chart_id="chart_002",
    chart_type=ChartType.LINE,
    title="Temperature Comparison: Hanoi vs Ho Chi Minh City",
    x_axis_label="Month",
    y_axis_label="Temperature (Celsius)",
    series=[
        DataSeries(
            name="Hanoi",
            color=Color(r=219, g=68, b=55),
            points=[
                DataPoint(label="Jan", value=17.2, confidence=0.90),
                DataPoint(label="Feb", value=18.5, confidence=0.91),
                DataPoint(label="Mar", value=21.3, confidence=0.88),
                DataPoint(label="Apr", value=24.8, confidence=0.92),
                DataPoint(label="May", value=28.1, confidence=0.89),
                DataPoint(label="Jun", value=29.5, confidence=0.93),
                DataPoint(label="Jul", value=29.8, confidence=0.87),
                DataPoint(label="Aug", value=29.0, confidence=0.90),
                DataPoint(label="Sep", value=27.2, confidence=0.85),
                DataPoint(label="Oct", value=24.5, confidence=0.91),
                DataPoint(label="Nov", value=21.0, confidence=0.88),
                DataPoint(label="Dec", value=18.0, confidence=0.90),
            ],
        ),
        DataSeries(
            name="Ho Chi Minh City",
            color=Color(r=15, g=157, b=88),
            points=[
                DataPoint(label="Jan", value=26.8, confidence=0.91),
                DataPoint(label="Feb", value=27.5, confidence=0.90),
                DataPoint(label="Mar", value=28.9, confidence=0.89),
                DataPoint(label="Apr", value=30.1, confidence=0.92),
                DataPoint(label="May", value=29.5, confidence=0.88),
                DataPoint(label="Jun", value=28.3, confidence=0.87),
                DataPoint(label="Jul", value=27.9, confidence=0.91),
                DataPoint(label="Aug", value=27.8, confidence=0.90),
                DataPoint(label="Sep", value=27.5, confidence=0.86),
                DataPoint(label="Oct", value=27.2, confidence=0.89),
                DataPoint(label="Nov", value=27.0, confidence=0.90),
                DataPoint(label="Dec", value=26.5, confidence=0.88),
            ],
        ),
    ],
    description="Line chart comparing monthly temperatures between Hanoi and Ho Chi Minh City. Hanoi shows strong seasonal variation (17-30C) while HCMC remains relatively stable (26-30C).",
    correction_log=["Corrected 'Ho Chi Minh' from OCR 'Ho Chl Mlnh'"],
)

print(f"Chart 2: {line_chart.title}")
print(f"  Type: {line_chart.chart_type.value}")
print(f"  Series: {len(line_chart.series)}")
for s in line_chart.series:
    print(f"    - {s.name}: {s.count} points")

In [ ]:
# -- Chart 3: Pie chart with anomaly (one slice dominates) --
pie_chart = RefinedChartData(
    chart_id="chart_003",
    chart_type=ChartType.PIE,
    title="Market Share by Company",
    x_axis_label=None,
    y_axis_label="Percentage (%)",
    series=[
        DataSeries(
            name="Market Share",
            points=[
                DataPoint(label="Company A", value=52.3, unit="%", confidence=0.94),
                DataPoint(label="Company B", value=18.7, unit="%", confidence=0.91),
                DataPoint(label="Company C", value=12.4, unit="%", confidence=0.89),
                DataPoint(label="Company D", value=9.8, unit="%", confidence=0.87),
                DataPoint(label="Others", value=6.8, unit="%", confidence=0.85),
            ],
        ),
    ],
    description="Pie chart showing market share distribution. Company A dominates with 52.3%, followed by Company B (18.7%), Company C (12.4%), Company D (9.8%), and Others (6.8%).",
    correction_log=[],
)

print(f"Chart 3: {pie_chart.title}")
print(f"  Type: {pie_chart.chart_type.value}")
print(f"  Slices: {pie_chart.series[0].count}")

In [ ]:
# -- Assemble Stage4Output --
stage4_output = Stage4Output(
    session=session,
    charts=[bar_chart, line_chart, pie_chart],
)

print(f"Stage4Output assembled:")
print(f"  Session: {stage4_output.session.session_id}")
print(f"  Charts:  {len(stage4_output.charts)}")
for c in stage4_output.charts:
    total_points = sum(s.count for s in c.series)
    print(f"    [{c.chart_id}] {c.chart_type.value:10s} | {c.title} | {total_points} data points")

## 2. Run Stage 5 with Default Config

`Stage5Reporting` takes `Stage4Output` and produces `PipelineResult`.

Key operations:
1. Wraps each `RefinedChartData` into `FinalChartResult`
2. Generates insights (trend, comparison, anomaly, summary) for each chart
3. Creates processing summary
4. Optionally saves JSON, Markdown, CSV reports

In [ ]:
# -- Create Stage5 with default config, but disable file saving for notebook use --
config = ReportingConfig(
    enable_insights=True,
    max_insights_per_chart=5,
    min_series_for_comparison=2,
    anomaly_z_score_threshold=2.0,
    save_json=False,     # Disable file I/O for notebook
    save_report=False,
    save_markdown=False,
    save_csv=False,
)

stage5 = Stage5Reporting(config=config)
print(f"Stage5Reporting initialized")
print(f"  enable_insights: {config.enable_insights}")
print(f"  max_insights_per_chart: {config.max_insights_per_chart}")
print(f"  anomaly_z_score_threshold: {config.anomaly_z_score_threshold}")

In [ ]:
# -- Run Stage 5 --
result = stage5.process(stage4_output)

print(f"=== PipelineResult ===")
print(f"  Session:          {result.session.session_id}")
print(f"  Total charts:     {result.total_charts}")
print(f"  Chart types:      {result.chart_types_summary}")
print(f"  Processing time:  {result.processing_time_seconds:.3f}s")
print(f"  Summary:          {result.summary[:120]}...")
print(f"  Warnings:         {len(result.warnings)}")
print(f"  Model versions:   {result.model_versions}")

## 3. Inspect FinalChartResult Structure

Each chart in the output is a `FinalChartResult`:
- `data`: The original `RefinedChartData` (passed through)
- `insights`: List of `ChartInsight` generated by Stage 5
- `source_info`: Traceability metadata

In [ ]:
# -- Inspect each chart result --
for i, chart_result in enumerate(result.charts):
    print(f"\n{'='*60}")
    print(f"Chart {i+1}: [{chart_result.chart_id}] {chart_result.chart_type.value}")
    print(f"  Title: {chart_result.title}")
    print(f"  Source info: {chart_result.source_info}")
    print(f"  Insights ({len(chart_result.insights)}):")
    for j, insight in enumerate(chart_result.insights):
        print(f"    [{j+1}] {insight.insight_type:12s} | conf={insight.confidence:.2f} | {insight.text[:80]}")
    
    # Verify data passthrough
    print(f"  Data series: {len(chart_result.data.series)}")
    for s in chart_result.data.series:
        print(f"    - {s.name}: {s.count} points")

## 4. Deep Dive: Insight Types

Stage 5 generates 4 types of insights:

| Type | Algorithm | When Generated |
|------|-----------|----------------|
| **summary** | Descriptive statistics | Always (for every chart) |
| **trend** | Linear regression on ordered data | When series has 3+ numeric points |
| **comparison** | Cross-series analysis | When chart has 2+ series |
| **anomaly** | Z-score outlier detection | When z-score > threshold |

In [ ]:
# -- Group insights by type across all charts --
from collections import defaultdict

insights_by_type = defaultdict(list)
for chart_result in result.charts:
    for insight in chart_result.insights:
        insights_by_type[insight.insight_type].append({
            "chart": chart_result.chart_id,
            "text": insight.text,
            "confidence": insight.confidence,
        })

print(f"Insight distribution across all charts:")
for itype, items in sorted(insights_by_type.items()):
    print(f"\n  {itype.upper()} ({len(items)} insights):")
    for item in items:
        print(f"    [{item['chart']}] conf={item['confidence']:.2f} | {item['text'][:90]}")

In [ ]:
# -- Focus on the bar chart (chart_001): Expect TREND insight --
bar_result = result.charts[0]
print(f"Bar chart insights for '{bar_result.title}':")
for insight in bar_result.insights:
    print(f"\n  Type: {insight.insight_type}")
    print(f"  Confidence: {insight.confidence:.2f}")
    print(f"  Text: {insight.text}")

In [ ]:
# -- Focus on the line chart (chart_002): Expect COMPARISON + TREND --
line_result = result.charts[1]
print(f"Line chart insights for '{line_result.title}':")
for insight in line_result.insights:
    print(f"\n  Type: {insight.insight_type}")
    print(f"  Confidence: {insight.confidence:.2f}")
    print(f"  Text: {insight.text}")

In [ ]:
# -- Focus on the pie chart (chart_003): Expect ANOMALY (Company A dominates) --
pie_result = result.charts[2]
print(f"Pie chart insights for '{pie_result.title}':")
for insight in pie_result.insights:
    print(f"\n  Type: {insight.insight_type}")
    print(f"  Confidence: {insight.confidence:.2f}")
    print(f"  Text: {insight.text}")

## 5. Serialization: JSON Output

View the complete PipelineResult as JSON -- this is the final output format.

In [ ]:
# -- Full JSON serialization --
result_json = result.model_dump(mode="json")

# Show top-level keys
print(f"Top-level keys: {list(result_json.keys())}")
print(f"\nJSON size: {len(json.dumps(result_json)):,} characters")
print(f"\n--- First chart (truncated) ---")
first_chart = result_json["charts"][0]
print(json.dumps(first_chart, indent=2, default=str)[:800])

In [ ]:
# -- Save to file for inspection --
output_path = PROJECT_ROOT / "data" / "output" / "nb05_stage5_demo.json"
output_path.parent.mkdir(parents=True, exist_ok=True)
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(result_json, f, indent=2, default=str)

print(f"Saved to: {output_path}")
print(f"File size: {output_path.stat().st_size:,} bytes")

## 6. Config Tuning Experiments

Explore how `ReportingConfig` parameters affect output.

In [ ]:
# -- Experiment 1: Disable insights --
config_no_insights = ReportingConfig(
    enable_insights=False,
    save_json=False, save_report=False, save_markdown=False,
)
stage5_no_insights = Stage5Reporting(config=config_no_insights)
result_no_insights = stage5_no_insights.process(stage4_output)

print(f"With insights disabled:")
for c in result_no_insights.charts:
    print(f"  [{c.chart_id}] insights: {len(c.insights)}")
print(f"  -> No insights generated (as expected)")

In [ ]:
# -- Experiment 2: Stricter anomaly threshold --
config_strict = ReportingConfig(
    enable_insights=True,
    anomaly_z_score_threshold=1.0,  # Lower = more anomalies detected
    max_insights_per_chart=10,
    save_json=False, save_report=False, save_markdown=False,
)
stage5_strict = Stage5Reporting(config=config_strict)
result_strict = stage5_strict.process(stage4_output)

print(f"With strict anomaly threshold (z=1.0):")
for c in result_strict.charts:
    anomalies = [i for i in c.insights if i.insight_type == "anomaly"]
    print(f"  [{c.chart_id}] total insights: {len(c.insights)}, anomalies: {len(anomalies)}")
    for a in anomalies:
        print(f"    -> {a.text[:80]}")

In [ ]:
# -- Experiment 3: Limit insights per chart --
config_limited = ReportingConfig(
    enable_insights=True,
    max_insights_per_chart=2,  # Only 2 insights per chart
    save_json=False, save_report=False, save_markdown=False,
)
stage5_limited = Stage5Reporting(config=config_limited)
result_limited = stage5_limited.process(stage4_output)

print(f"With max_insights_per_chart=2:")
for c in result_limited.charts:
    print(f"  [{c.chart_id}] insights: {len(c.insights)} (max=2)")
    for i in c.insights:
        print(f"    - {i.insight_type}: {i.text[:70]}")

## 7. Data Contract Summary

Verify the exact shape of Stage 5 input/output.

In [ ]:
# -- Stage 4 -> Stage 5 data contract --
print("=" * 70)
print("STAGE 5 DATA CONTRACT")
print("=" * 70)

print("\n--- INPUT: Stage4Output ---")
print(f"  session: SessionInfo")
print(f"  charts:  List[RefinedChartData]  (count={len(stage4_output.charts)})")
for c in stage4_output.charts:
    print(f"    [{c.chart_id}] {c.chart_type.value}")
    print(f"      title: {c.title}")
    print(f"      x_axis_label: {c.x_axis_label}")
    print(f"      y_axis_label: {c.y_axis_label}")
    print(f"      series: {len(c.series)} (total points: {sum(s.count for s in c.series)})")
    print(f"      description: {c.description[:60]}...")
    print(f"      correction_log: {c.correction_log}")

print("\n--- OUTPUT: PipelineResult ---")
print(f"  session:                 SessionInfo")
print(f"  charts:                  List[FinalChartResult]  (count={result.total_charts})")
print(f"  summary:                 {result.summary[:60]}...")
print(f"  processing_time_seconds: {result.processing_time_seconds:.3f}")
print(f"  model_versions:          {result.model_versions}")
print(f"  warnings:                {result.warnings}")
print(f"  total_charts:            {result.total_charts}  (computed)")
print(f"  chart_types_summary:     {result.chart_types_summary}  (computed)")

print("\n--- FinalChartResult structure ---")
fcr = result.charts[0]
print(f"  chart_id:    {fcr.chart_id}")
print(f"  chart_type:  {fcr.chart_type}")
print(f"  title:       {fcr.title}")
print(f"  data:        RefinedChartData (passthrough from Stage 4)")
print(f"  insights:    List[ChartInsight] (count={len(fcr.insights)})")
print(f"  source_info: {fcr.source_info}")

In [ ]:
# -- ChartInsight structure --
print("--- ChartInsight structure ---")
if result.charts[0].insights:
    sample = result.charts[0].insights[0]
    print(f"  insight_type: {sample.insight_type}")
    print(f"  text:         {sample.text}")
    print(f"  confidence:   {sample.confidence}")
else:
    print("  (no insights to show)")

## Summary

**Stage 5 transforms**:
- `Stage4Output` (refined data with series/points) -> `PipelineResult` (final output with insights + summary)

**Key behaviors**:
1. Each `RefinedChartData` is wrapped into `FinalChartResult` with added `insights` and `source_info`
2. Insight generation uses statistical algorithms (linear regression for trends, z-score for anomalies)
3. `ReportingConfig` controls insight generation, thresholds, and output formats
4. Output is a self-contained `PipelineResult` with full traceability